# Experiment D: Longitudinal Engagement Decay (WildChat)

## Purpose
Extends Experiment C from 4 sessions (days) to months-long timescales using real-world human–AI conversation data.

## Data
**WildChat-1M** (Allen AI). 10K conversation sample from repeat users (≥5 conversations each).

## Analyses
1. **Longitudinal decay**: same methodology as Exp C but over months
2. **Turn complexity decay**: words/turn, lexical diversity over time
3. **Topic drift**: vocabulary concentration changes

In [ ]:
import sys
sys.path.insert(0, '..')
sys.path.insert(0, '../..')

import matplotlib.pyplot as plt
import pandas as pd
from experiments.shared.plotting import setup_style
setup_style()

## 1. Load WildChat Sample

In [ ]:
from experiments.shared.data_acquisition import load_wildchat_sample
from experiments.exp_d_wildchat_decay.analysis import prepare_wildchat, build_user_conversation_index

raw_df = load_wildchat_sample()
df = prepare_wildchat(raw_df)
print(f"{len(df)} turns, {df['conversation_hash'].nunique()} conversations, {df['ip_hash'].nunique()} users")

conv_index = build_user_conversation_index(df)
print(f"\nConversations per user: {conv_index.groupby('ip_hash')['conv_order'].max().describe()}")

## 2. Engagement Metrics Over Time

In [ ]:
from experiments.exp_d_wildchat_decay.analysis import compute_conversation_metrics

conv_metrics = compute_conversation_metrics(df)
conv_metrics.head()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
metrics = [
    ('passive_rate', 'Passive Rate'),
    ('evaluative_rate', 'Evaluative Rate'),
    ('mean_words_per_turn', 'Mean Words/Turn'),
    ('mean_ttr', 'Lexical Diversity (TTR)'),
]
for ax, (metric, title) in zip(axes.flat, metrics):
    conv_metrics['order_bin'] = pd.qcut(conv_metrics['conv_order'], q=10, labels=False, duplicates='drop')
    binned = conv_metrics.groupby('order_bin')[metric].agg(['mean', 'sem'])
    ax.errorbar(binned.index + 1, binned['mean'], yerr=binned['sem'], marker='o', capsize=3)
    ax.set_title(title)
    ax.set_xlabel('Conversation Order Decile')
fig.suptitle('Engagement Metrics Over Conversation History', fontweight='bold')
fig.tight_layout()
plt.show()

## 3. Decay Slopes

In [ ]:
from experiments.exp_d_wildchat_decay.analysis import compute_decay_slopes

print("By conversation order:")
decay_order = compute_decay_slopes(conv_metrics, time_col='conv_order')
display(decay_order)

print("\nBy weeks since first conversation:")
decay_weeks = compute_decay_slopes(conv_metrics, time_col='weeks_since_first')
display(decay_weeks)

## 4. Topic Drift

In [ ]:
from experiments.exp_d_wildchat_decay.analysis import compute_topic_diversity
from experiments.shared.stats_utils import per_subject_slopes, slope_ttest

topic_df = compute_topic_diversity(df)
topic_merged = topic_df.merge(
    conv_metrics[['ip_hash', 'conversation_hash', 'conv_order']],
    on=['ip_hash', 'conversation_hash'], how='left'
)
topic_slopes = per_subject_slopes(topic_merged, 'ip_hash', 'conv_order', 'vocab_concentration')
topic_tt = slope_ttest(topic_slopes)
print(f"Vocab concentration slope: {topic_tt['mean_slope']:+.5f} (t={topic_tt['t_stat']:.2f}, p={topic_tt['p_value']:.4f})")
print(f"{topic_tt['pct_declining']:.0%} of users show decreasing concentration")

## 5. Cross-Study Comparison

| Study | Timescale | Passive slope | Active slope |
|-------|-----------|--------------|-------------|
| Bastani (Exp C) | 4 sessions (days) | +0.027*** | -0.012** |
| WildChat (Exp D) | Months | (see above) | (see above) |

The longer timescale is expected to show clearer decay patterns, consistent with Fan et al.'s multi-week study and WildChat's month-long trajectories.

## 6. Interpretation

The WildChat analysis extends the CCL's argument for sustained scaffolding:
- If engagement decays over months (not just sessions), then scaffolding must persist
- Topic drift toward simpler queries suggests users may converge on low-effort interaction patterns
- This directly informs the CCL's fading debate: scaffolding for metacognitive skills should not be faded the way task scaffolding is